# Universal_test_scraper 

In [ ]:
import requests
from urllib.parse import urlparse

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

session = requests.Session()
session.headers.update(HEADERS)


# ============ CLEAN URL ============
def clean_url(url):
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}"


# ============ SHOPIFY CHECK ============
def is_shopify(store_url):
    try:
        r = session.get(f"{store_url}/products.json", timeout=10)
        return r.status_code == 200 and "products" in r.json()
    except:
        return False


# ============ SHOPIFY TEST ============
def test_shopify(store_url):
    r = session.get(f"{store_url}/products.json?limit=5")
    products = r.json().get("products", [])

    print(f"🟢 Shopify Products Found: {len(products)}")
    for p in products:
        print("—", p["title"])


# ============ HTML TEST ============
def test_html(store_url):
    r = session.get(store_url, timeout=10)
    if len(r.text) < 2000:
        print("🔴 JS Heavy Website")
        return

    print("🟡 HTML page loaded")
    if "product" in r.text.lower():
        print("🟡 Product keywords detected")
    else:
        print("⚠️ No product structure detected")


# ============ MAIN ============
def universal_test(url):
    print("\n==============================")
    store = clean_url(url)
    print("🔍 Testing:", store)

    if is_shopify(store):
        print("✅ Platform: SHOPIFY")
        test_shopify(store)
    else:
        print("🟡 Platform: NON-SHOPIFY")
        test_html(store)


# ===== CHANGE ONLY THIS =====
URL = "https://books.toscrape.com/"
universal_test(URL)


# Universal_final_scraper

In [ ]:
import requests
import pandas as pd
from urllib.parse import urlparse
from datetime import datetime
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

session = requests.Session()
session.headers.update(HEADERS)


# ============ CLEAN URL ============
def clean_url(url):
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}"


# ============ SHOPIFY CHECK ============
def is_shopify(store_url):
    try:
        r = session.get(f"{store_url}/products.json", timeout=10)
        return r.status_code == 200 and "products" in r.json()
    except:
        return False


# ============ SHOPIFY SCRAPER ============
def scrape_shopify(store_url):
    print("🟢 Shopify JSON scraping started")
    all_products = []
    page = 1

    while True:
        api = f"{store_url}/products.json?limit=250&page={page}"
        r = session.get(api)
        if r.status_code != 200:
            break

        products = r.json().get("products", [])
        if not products:
            break

        for p in products:
            prices = [float(v["price"]) for v in p["variants"] if v["price"]]
            price = min(prices) if prices else ""

            stock = "In Stock" if any(v["available"] for v in p["variants"]) else "Sold Out"

            all_products.append({
                "Product Name": p["title"],
                "Vendor": p.get("vendor"),
                "Category": p.get("product_type"),
                "Price": price,
                "Stock": stock,
                "Tags": ", ".join(p.get("tags", [])),
                "Images": ", ".join(i["src"] for i in p["images"]),
                "Product URL": f"{store_url}/products/{p['handle']}"
            })

        page += 1

    return all_products


# ============ HTML SCRAPER (SAFE MODE) ============
def scrape_html(store_url):
    print("🟡 Limited HTML scraping")

    r = session.get(store_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    products = []
    links = soup.select("a[href*='/product'], a[href*='/products/']")

    for a in links[:40]:
        name = a.get_text(strip=True)
        href = a.get("href")

        if not name or len(name) < 5:
            continue

        url = href if href.startswith("http") else store_url + href

        products.append({
            "Product Name": name,
            "Vendor": "",
            "Category": "",
            "Price": "",
            "Stock": "",
            "Tags": "",
            "Images": "",
            "Product URL": url
        })

    return products


# ============ MAIN ============
def universal_scraper(url):
    print("\n==============================")
    store = clean_url(url)
    print("🔍 Scraping:", store)

    if is_shopify(store):
        data = scrape_shopify(store)
    else:
        data = scrape_html(store)

        if len(data) < 5:
            print("🔴 JS Heavy Website — Selenium/Playwright required")
            return

    if not data:
        print("❌ No data scraped")
        return

    df = pd.DataFrame(data)
    filename = f"products_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    df.to_excel(filename, index=False)

    print(f"✅ Saved: {filename}")
    print(f"📦 Total Products: {len(df)}")


# ===== CHANGE ONLY THIS =====
URL = "https://www.nicobar.com"
universal_scraper(URL)